# Nuclear Waste Canister Temperature Prediction — Random Forest
**CIVIL-226 - Introduction to Machine Learning for Engineers — EPFL**

**Members:** Nour NAJA | Alessandro CLERICI

## Objective
Predict temperature around nuclear waste canisters at unobserved sensor positions using a Random Forest regressor with physics-motivated features.

The dataset contains 3 experimental replications per sensor/timestep. The test set corresponds exclusively to Rep 2 (radioactive decay 1488W→299W), so we train only on Rep 2 data for physical consistency.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

## 2. Load Data

In [ ]:
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')
sensors = pd.read_parquet('data/sensors.parquet')

# Remove duplicate sensors (N206, N213 — identical coordinates, reporting error)
n_before = len(sensors)
sensors  = sensors.drop_duplicates(subset='sensor', keep='first').reset_index(drop=True)
print(f'Duplicate sensors removed: {n_before - len(sensors)} (N206, N213)')
print(f'Sensors : {sensors.shape}  ->  {sensors.columns.tolist()}')
print(f'Train   : {train.shape}   ->  {train.columns.tolist()}')
print(f'Test    : {test.shape}    ->  {test.columns.tolist()}')

## 3. Merge Sensor Positions

In [ ]:
train_full = train.merge(sensors, on='sensor', how='left')
print(train_full.head())
print(train_full.shape)

## 4. Clean Missing Values

In [ ]:
print(f'Rows before NaN removal: {len(train_full)}')
train_full = train_full.dropna(subset=['temperature']).copy()
print(f'Rows after NaN removal : {len(train_full)}')

## 5. Data Exploration

In [ ]:
print(train_full['temperature'].describe(percentiles=[0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]))
print('Temperature min:', train_full['temperature'].min())
print('Temperature max:', train_full['temperature'].max())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_full['temperature'].dropna(), bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Temperature distribution')
axes[0].set_xlabel('Temperature [°C]')
axes[0].set_ylabel('Frequency')

temp_by_time = train_full.groupby('time')['temperature'].mean()
axes[1].plot(temp_by_time.index / (365.25*24*3600), temp_by_time.values, color='tomato', linewidth=0.8)
axes[1].set_title('Mean temperature over time')
axes[1].set_xlabel('Time [years]')
axes[1].set_ylabel('Mean temperature [°C]')
plt.tight_layout()
plt.show()

# Sensor positions
train_sensors = set(train_full['sensor'].unique())
test_sensors  = set(test['sensor'].unique())
s_train = sensors[sensors['sensor'].isin(train_sensors)]
s_test  = sensors[sensors['sensor'].isin(test_sensors)]

plt.figure(figsize=(8, 4))
plt.scatter(s_train['coor_x'], s_train['coor_y'], c='steelblue', s=20, alpha=0.6, label='Train sensors')
plt.scatter(s_test['coor_x'],  s_test['coor_y'],  c='tomato',    s=40, alpha=0.9, marker='^', label='Test sensors')
plt.axvline(1.4, color='gray', linestyle='--', alpha=0.5, label='Buffer/OPA boundary')
plt.xlabel('coor_x [m]'); plt.ylabel('coor_y [m]')
plt.title('Sensor positions (2D)')
plt.legend(); plt.tight_layout(); plt.show()

## 6. Outlier Removal

Two complementary methods:
- **Global MAD** (Median Absolute Deviation): catches sensors with physically impossible temperatures
- **Local IQR per timestep**: catches values deviating too much from the distribution at a given moment in time

In [ ]:
# Global outliers via MAD
temp        = train_full['temperature']
median_temp = temp.median()
mad_temp    = np.median(np.abs(temp - median_temp))
robust_z    = 0.6745 * (temp - median_temp) / (mad_temp + 1e-8)
train_full['is_global_outlier'] = robust_z.abs() > 6
print(f'Global outliers (MAD): {train_full["is_global_outlier"].sum()}')

# Local outliers per timestep via IQR
time_stats = (
    train_full.groupby('time')['temperature']
    .agg(time_median='median',
         time_q1=lambda x: x.quantile(0.25),
         time_q3=lambda x: x.quantile(0.75))
    .reset_index()
)
time_stats['time_iqr'] = time_stats['time_q3'] - time_stats['time_q1']
train_full = train_full.merge(time_stats, on='time', how='left')
train_full['is_local_outlier'] = (
    (train_full['temperature'] < train_full['time_median'] - 4.0 * train_full['time_iqr']) |
    (train_full['temperature'] > train_full['time_median'] + 4.0 * train_full['time_iqr'])
)
print(f'Local outliers (IQR per timestep): {train_full["is_local_outlier"].sum()}')

before_rows = len(train_full)
train_full  = train_full[
    (~train_full['is_global_outlier']) & (~train_full['is_local_outlier'])
].copy()
print(f'Rows removed: {before_rows - len(train_full)} | Remaining: {len(train_full)}')

cols_to_drop = ['is_global_outlier', 'is_local_outlier',
                'time_median', 'time_q1', 'time_q3', 'time_iqr']
train_full = train_full.drop(columns=[c for c in cols_to_drop if c in train_full.columns])

## 7. Sensor Drift Detection & Correction

Some sensors show a systematic linear drift in their residuals over time. We detect and correct this before training.

In [ ]:
time_median = (
    train_full.groupby('time')['temperature']
    .median().rename('global_time_median').reset_index()
)
drift_df = train_full.merge(time_median, on='time', how='left').copy()
drift_df['temp_residual'] = drift_df['temperature'] - drift_df['global_time_median']

t_min = drift_df['time'].min()
t_max = drift_df['time'].max()
drift_df['time_norm_for_drift'] = (drift_df['time'] - t_min) / (t_max - t_min + 1e-8)

drift_records = []
for sensor_id, g in drift_df.groupby('sensor'):
    g = g.sort_values('time_norm_for_drift')
    if len(g) < 20:
        continue
    x = g['time_norm_for_drift'].values
    y = g['temp_residual'].values
    slope = np.polyfit(x, y, 1)[0] if np.std(y) > 1e-8 else 0.0
    corr  = np.corrcoef(x, y)[0, 1] if np.std(y) > 1e-8 else 0.0
    drift_records.append({'sensor': sensor_id, 'drift_slope': slope, 'drift_corr': corr})

sensor_drift    = pd.DataFrame(drift_records)
slope_abs       = sensor_drift['drift_slope'].abs()
slope_threshold = slope_abs.median() + 4 * (np.median(np.abs(slope_abs - slope_abs.median())) + 1e-8)
sensor_drift['is_drift_sensor'] = (
    (sensor_drift['drift_slope'].abs() > slope_threshold) &
    (sensor_drift['drift_corr'].abs() > 0.5)
)
drift_sensors = sensor_drift.loc[sensor_drift['is_drift_sensor'], 'sensor'].unique()
print(f'Drift sensors detected: {len(drift_sensors)}')

t_min = train_full['time'].min()
t_max = train_full['time'].max()
train_full['time_norm_for_drift'] = (train_full['time'] - t_min) / (t_max - t_min + 1e-8)
drift_map = sensor_drift.set_index('sensor')['drift_slope'].to_dict()
train_full['drift_correction'] = 0.0
for s in drift_sensors:
    mask = train_full['sensor'] == s
    train_full.loc[mask, 'drift_correction'] = (
        drift_map[s] * (train_full.loc[mask, 'time_norm_for_drift'] - 0.5)
    )
train_full['temperature'] = train_full['temperature'] - train_full['drift_correction']
train_full = train_full.drop(columns=['time_norm_for_drift', 'drift_correction'])
print('Drift correction applied.')

## 8. Filter Replication 2

Each (sensor, timestep) has exactly 3 rows — one per experimental replication:
- **Rep 0**: 500W → 0W (drops around year 100)
- **Rep 1**: 1400W → 0W (drops around year 110)
- **Rep 2**: radioactive decay 1488W → 299W over 250 years

The test set matches **Rep 2 exclusively**. Training on Reps 0 and 1 would introduce noise from physically incompatible power scenarios.

In [ ]:
train_full['rep'] = (
    train_full.groupby(['sensor', 'time'])['power']
    .rank(method='first').astype(int) - 1
)
for rep in [0, 1, 2]:
    sub = train_full[train_full['rep'] == rep]
    print(f'Rep {rep}: {sub.sensor.nunique()} sensors, {len(sub):,} rows')

train_full = train_full[train_full['rep'] == 2].drop(columns=['rep']).copy()
print(f'\nAfter filtering Rep 2: {len(train_full):,} rows, {train_full.sensor.nunique()} sensors')

## 9. Feature Engineering

Physics-motivated features:
- **dist_canister**: temperature decays with distance from the heat source
- **is_opa**: zone indicator (OPA weighted more in Kaggle score)
- **time_log**: captures logarithmic thermal diffusion dynamics
- **power_x_time, dist_x_time**: physical interactions between power, distance and time
- **inv_dist_canister**: radial gradient near the canister
- **coor_x_squared**: spatial non-linearity in x
- **cum_energy**: total energy injected into the system

In [ ]:
eps       = 1e-8
t_max_ref = train_full['time'].max()

def add_cum_energy(df):
    """Compute cumulative energy (power × dt) and merge back."""
    power_time = (
        df[['time', 'power']]
        .drop_duplicates(subset=['time'])
        .sort_values('time').copy()
    )
    power_time['dt']         = power_time['time'].diff().fillna(0)
    power_time['cum_energy'] = (power_time['power'] * power_time['dt']).cumsum()
    return df.merge(
        power_time[['time', 'cum_energy']].drop_duplicates(subset=['time']),
        on='time', how='left'
    )

def add_features(df, t_max_ref):
    """
    Add physics-motivated engineered features.
    t_max_ref: max time from training data for consistent time_norm on test.
    """
    df = df.copy()

    # Spatial features
    df['dist_center']       = np.sqrt(df['coor_x']**2 + df['coor_y']**2)
    df['dist_canister']     = np.sqrt((df['coor_x'] - 0.7)**2 + (df['coor_y'] - 1.2)**2)
    df['is_opa']            = (df['coor_x'] > 1.4).astype(float)
    df['inv_dist_canister'] = 1 / (df['dist_canister'] + 0.1)
    df['coor_x_squared']    = df['coor_x'] ** 2

    # Temporal features
    df['time_norm'] = df['time'] / t_max_ref
    df['time_log']  = np.log1p(df['time'])

    # Physical interaction features
    df['power_x_time'] = df['power'] * df['time_norm']
    df['dist_x_time']  = df['dist_canister'] * df['time_norm']
    df['x_x_time']     = df['coor_x'] * df['time_norm']
    df['dist_x_log']   = df['dist_canister'] * df['time_log']
    df['time_x_dist']  = df['time_norm'] * df['dist_canister']

    return df

train_full = add_cum_energy(train_full)
train_full = add_features(train_full, t_max_ref)
print('Features after engineering:', train_full.columns.tolist())

## 10. Final Checks

In [ ]:
print('Final train_full shape:', train_full.shape)
print('Missing values:')
print(train_full.isna().sum())
assert train_full['temperature'].notna().all()
assert np.isfinite(train_full['temperature']).all()
print('All checks passed.')

## 11. Train/Validation Split by Sensor

We split by sensor (not by row) to properly evaluate generalization to unseen sensor positions — the actual test scenario. A row-based split would leak spatial information and give an over-optimistic RMSE.

In [ ]:
TARGET   = 'temperature'
FEATURES = [
    'coor_x', 'coor_y',
    'time_norm', 'time_log',
    'power', 'cum_energy',
    'dist_center', 'dist_canister', 'is_opa',
    'inv_dist_canister', 'coor_x_squared',
    'power_x_time', 'dist_x_time', 'x_x_time', 'dist_x_log', 'time_x_dist'
]

unique_sensors = train_full['sensor'].unique()
rng            = np.random.default_rng(42)
val_sensors    = rng.choice(unique_sensors, size=int(0.2 * len(unique_sensors)), replace=False)
train_sensors  = np.setdiff1d(unique_sensors, val_sensors)

train_df = train_full[train_full['sensor'].isin(train_sensors)].copy()
val_df   = train_full[train_full['sensor'].isin(val_sensors)].copy()

assert set(train_df['sensor']).isdisjoint(set(val_df['sensor']))
print(f'Train sensors: {train_df.sensor.nunique()} | Rows: {len(train_df):,}')
print(f'Val   sensors: {val_df.sensor.nunique()}   | Rows: {len(val_df):,}')

# Visualize train/val split
train_sensor_pos = train_df.groupby('sensor')[['coor_x', 'coor_y']].mean().reset_index()
val_sensor_pos   = val_df.groupby('sensor')[['coor_x', 'coor_y']].mean().reset_index()

plt.figure(figsize=(10, 4))
plt.scatter(train_sensor_pos['coor_x'], train_sensor_pos['coor_y'], s=35, label='train', alpha=0.7)
plt.scatter(val_sensor_pos['coor_x'],   val_sensor_pos['coor_y'],   s=80, label='validation', alpha=0.9, marker='x')
plt.axvline(1.4, color='gray', linestyle='--', alpha=0.5, label='Buffer/OPA boundary')
plt.xlabel('coor_x'); plt.ylabel('coor_y')
plt.title('Spatial distribution of train/validation sensors')
plt.legend(loc='upper right'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 12. Normalisation

In [ ]:
X_train = train_df[FEATURES].values
y_train = train_df[TARGET].values
X_val   = val_df[FEATURES].values
y_val   = val_df[TARGET].values

# Fit scaler ONLY on training data to avoid data leakage
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

print(f'X_train: {X_train_s.shape} | X_val: {X_val_s.shape}')
print(f'NaN/Inf in X_train_s: {np.isnan(X_train_s).sum()} / {np.isinf(X_train_s).sum()}')
print(f'NaN/Inf in X_val_s  : {np.isnan(X_val_s).sum()} / {np.isinf(X_val_s).sum()}')

## 13. Model — Random Forest

Random Forest trains T independent decision trees on different data subsamples and averages their predictions:

$$\hat{y} = \frac{1}{T}\sum_{t=1}^{T} h_t(\mathbf{x})$$

This ensemble approach reduces variance compared to a single tree and captures non-linear interactions between position, time and power. `max_samples=0.3` means each tree sees 30% of the data — proper bagging without a manual subsample.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    max_samples=0.3,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_s, y_train)
print('Training done.')

## 14. Evaluate on Validation Set

In [ ]:
y_val_pred = rf.predict(X_val_s)

mae  = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2   = r2_score(y_val, y_val_pred)

print(f'Validation MAE  : {mae:.4f} °C')
print(f'Validation RMSE : {rmse:.4f} °C')
print(f'Validation R²   : {r2:.4f}')

# Train RMSE to check for overfitting
y_train_pred = rf.predict(X_train_s)
rmse_train   = np.sqrt(mean_squared_error(y_train, y_train_pred))
print(f'\nTrain RMSE : {rmse_train:.4f} °C  (gap = {rmse - rmse_train:.4f})')

In [ ]:
# True vs predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(y_val, y_val_pred, s=3, alpha=0.3)
mn, mx = min(y_val.min(), y_val_pred.min()), max(y_val.max(), y_val_pred.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('True temperature [°C]')
plt.ylabel('Predicted temperature [°C]')
plt.title('Random Forest — True vs Predicted on unseen sensors')
plt.grid(True)
plt.tight_layout()
plt.show()

## 15. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
importances.plot(kind='barh', figsize=(7, 6), color='steelblue')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 16. Error Analysis by Sensor

In [ ]:
val_results = val_df.copy()
val_results['y_true']    = y_val
val_results['y_pred']    = y_val_pred
val_results['abs_error'] = np.abs(y_val - y_val_pred)
val_results['sq_error']  = (y_val - y_val_pred)**2

sensor_metrics = (
    val_results
    .groupby('sensor')
    .agg(
        mae  =('abs_error', 'mean'),
        rmse =('sq_error',  lambda x: np.sqrt(np.mean(x))),
        coor_x=('coor_x', 'first'),
        coor_y=('coor_y', 'first'),
    )
    .sort_values('rmse', ascending=False)
)
print('Worst sensors by RMSE:')
print(sensor_metrics.head(10))

# Spatial error map
plt.figure(figsize=(8, 5))
sc = plt.scatter(
    sensor_metrics['coor_x'], sensor_metrics['coor_y'],
    c=sensor_metrics['rmse'], cmap='hot_r', s=80
)
plt.colorbar(sc, label='RMSE [°C]')
plt.axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA boundary')
plt.xlabel('coor_x'); plt.ylabel('coor_y')
plt.title('Validation RMSE by sensor position')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 17. MAE over Time

In [ ]:
train_results = train_df.copy()
train_results['y_pred']    = rf.predict(X_train_s)
train_results['abs_error'] = np.abs(train_results['temperature'] - train_results['y_pred'])
val_results['abs_error']   = np.abs(val_results['y_true'] - val_results['y_pred'])

train_results['time_years'] = train_results['time'] / (365.25 * 24 * 3600)
val_results['time_years']   = val_results['time']   / (365.25 * 24 * 3600)

all_time_years = pd.concat([train_results['time_years'], val_results['time_years']])
time_bins = pd.qcut(all_time_years, q=20, duplicates='drop').cat.categories

train_time_error = (
    train_results
    .groupby(pd.cut(train_results['time_years'], bins=time_bins, include_lowest=True), observed=False)
    .agg(train_mae=('abs_error', 'mean')).reset_index()
)
val_time_error = (
    val_results
    .groupby(pd.cut(val_results['time_years'], bins=time_bins, include_lowest=True), observed=False)
    .agg(val_mae=('abs_error', 'mean')).reset_index()
)
train_time_error['time_mid'] = train_time_error['time_years'].apply(lambda x: x.mid)
val_time_error['time_mid']   = val_time_error['time_years'].apply(lambda x: x.mid)

plt.figure(figsize=(10, 5))
plt.plot(train_time_error['time_mid'], train_time_error['train_mae'], marker='o', label='Training')
plt.plot(val_time_error['time_mid'],   val_time_error['val_mae'],     marker='o', label='Validation')
plt.xlabel('Time [years]'); plt.ylabel('MAE [°C]')
plt.title('MAE over time: training vs validation')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## 18. Final Predictions & Submission

In [ ]:
# Build test_full with same pipeline
test_full = test.merge(sensors, on='sensor', how='left')
test_full = add_cum_energy(test_full)
test_full = add_features(test_full, t_max_ref)

X_test_s = scaler.transform(test_full[FEATURES].values)
assert len(X_test_s) == len(test)
print(f'X_test_s: {X_test_s.shape}')
print(f'Missing in test features: {np.isnan(X_test_s).sum()}')

y_pred = rf.predict(X_test_s)

submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})
assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'Random Forest — Validation RMSE: {rmse:.4f} °C')
print(f'submission.csv saved — {len(submission)} rows')
display(submission.head())

In [ ]:
# Sanity check: train vs test predictions over time
test_full['temp_pred'] = y_pred
temp_train = train_full.groupby('time')['temperature'].mean()
temp_pred  = test_full.groupby('time')['temp_pred'].mean()

plt.figure(figsize=(10, 4))
plt.plot(temp_train.index / (365.25*24*3600), temp_train.values,
         label='Train Rep2 (true)', color='tomato', alpha=0.7, linewidth=0.8)
plt.plot(temp_pred.index / (365.25*24*3600), temp_pred.values,
         label='Test (predicted)', color='steelblue', alpha=0.7, linewidth=0.8)
plt.title('Sanity check: Train vs Test predictions over time')
plt.xlabel('Time [years]'); plt.ylabel('Mean temperature [°C]')
plt.legend(); plt.tight_layout(); plt.show()

## 19. Spatial Temperature Map: True vs Predicted

In [ ]:
train_sensor_temp = (
    train_full.groupby('sensor')
    .agg(coor_x=('coor_x', 'first'), coor_y=('coor_y', 'first'), mean_temp=('temperature', 'mean'))
    .reset_index()
)
test_sensor_temp = (
    test_full.groupby('sensor')
    .agg(coor_x=('coor_x', 'first'), coor_y=('coor_y', 'first'), mean_temp=('temp_pred', 'mean'))
    .reset_index()
)

vmin = min(train_sensor_temp['mean_temp'].min(), test_sensor_temp['mean_temp'].min())
vmax = max(train_sensor_temp['mean_temp'].max(), test_sensor_temp['mean_temp'].max())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(
    train_sensor_temp['coor_x'], train_sensor_temp['coor_y'],
    c=train_sensor_temp['mean_temp'], cmap='hot_r', vmin=vmin, vmax=vmax,
    s=80, edgecolors='gray', linewidths=0.3
)
axes[0].axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA x=1.4')
axes[0].set_title('True temperatures — Train Rep2 sensors', fontsize=13)
axes[0].set_xlabel('coor_x [m]'); axes[0].set_ylabel('coor_y [m]')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
plt.colorbar(sc1, ax=axes[0], label='Mean temperature [°C]')

sc2 = axes[1].scatter(
    test_sensor_temp['coor_x'], test_sensor_temp['coor_y'],
    c=test_sensor_temp['mean_temp'], cmap='hot_r', vmin=vmin, vmax=vmax,
    s=80, edgecolors='gray', linewidths=0.3
)
axes[1].axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA x=1.4')
axes[1].set_title('Predicted temperatures — Test sensors (Random Forest)', fontsize=13)
axes[1].set_xlabel('coor_x [m]'); axes[1].set_ylabel('coor_y [m]')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
plt.colorbar(sc2, ax=axes[1], label='Mean temperature [°C]')

plt.suptitle('Spatial temperature distribution: true vs predicted\n'
             'Expected: radial gradient — hot near canister (x≈0), cool far away',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()